<a href="https://colab.research.google.com/github/Sumanth30-git/AI-Fake-Content-Detector/blob/main/Ai_Fake_Content_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch gradio


In [ ]:
from transformers import pipeline

detector = pipeline("text-classification", model="jy46604790/Fake-News-Bert-Detect")

print("Model loaded successfully!")

In [ ]:
def check_news(headline):
    result = detector(headline)[0]
    label = result['label']
    score = round(result['score'] * 100, 2)

    if label == "LABEL_1":
        verdict = "REAL NEWS"
    else:
        verdict = "FAKE NEWS"

    print(f"Headline : {headline}")
    print(f"Verdict  : {verdict}")
    print(f"Confidence: {score}%")
    print("-" * 50)

# Test it
check_news("Scientists confirm drinking coffee makes you immortal")
check_news("NASA launches new telescope to study distant galaxies")
check_news("Government announces new budget plan for 2025")

In [ ]:
!pip install timm Pillow requests


In [ ]:
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

# Load deepfake image detection model
img_detector = pipeline("image-classification",
                        model="dima806/deepfake_vs_real_image_detection")

print("Image model loaded!")


In [ ]:
from google.colab import files

def check_uploaded_image():
    print("Select an image file from your computer...")
    uploaded = files.upload()

    for filename in uploaded.keys():
        image = Image.open(BytesIO(uploaded[filename])).convert("RGB")

        results = img_detector(image)
        top = results[0]
        label = top['label']
        score = round(top['score'] * 100, 2)

        if label == "real":
            verdict = "✅ REAL IMAGE"
        else:
            verdict = "🚨 DEEPFAKE IMAGE"

        print(f"\nFile       : {filename}")
        print(f"Verdict    : {verdict}")
        print(f"Confidence : {score}%")
        print("-" * 50)

check_uploaded_image()

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
from transformers import pipeline
from PIL import Image

# Load both models
print("Loading models...")
news_detector = pipeline("text-classification", model="jy46604790/Fake-News-Bert-Detect")
img_detector = pipeline("image-classification", model="dima806/deepfake_vs_real_image_detection")
print("Both models ready!")

# Fake news function
def detect_fake_news(headline):
    if not headline.strip():
        return "Please enter a headline."
    result = news_detector(headline)[0]
    label = result['label']
    score = round(result['score'] * 100, 2)
    verdict = "🚨 FAKE NEWS" if label == "LABEL_0" else "✅ REAL NEWS"
    return f"{verdict}\nConfidence: {score}%"

def detect_deepfake(image):
    if image is None:
        return "Please upload an image."
    results = img_detector(image)
    top = results[0]
    label = top['label']  # 'Real' or 'Fake'
    score = round(top['score'] * 100, 2)

    if label == "Real":  # capital R
        verdict = "✅ REAL IMAGE"
    else:
        verdict = "🚨 DEEPFAKE IMAGE"

    return f"{verdict}\nConfidence: {score}%"

# Build UI
with gr.Blocks(title="AI Fake Content Detector") as app:
    gr.Markdown("# 🤖 AI-Based Fake Content Detection System")
    gr.Markdown("Detect deepfake images and fake news using AI")

    with gr.Tab("📰 Fake News Detection"):
        gr.Markdown("### Enter a news headline to check")
        news_input = gr.Textbox(placeholder="Type a news headline here...")
        news_button = gr.Button("Analyze", variant="primary")
        news_output = gr.Textbox(label="Result")
        news_button.click(detect_fake_news, inputs=news_input, outputs=news_output)

    with gr.Tab("🖼️ Deepfake Image Detection"):
        gr.Markdown("### Upload a face image to check")
        img_input = gr.Image(type="pil")
        img_button = gr.Button("Analyze", variant="primary")
        img_output = gr.Textbox(label="Result")
        img_button.click(detect_deepfake, inputs=img_input, outputs=img_output)

app.launch(share=True)

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load fake news dataset from HuggingFace
print("Downloading dataset...")
dataset = load_dataset("GonzaloA/fake_news", split="test")
df = pd.DataFrame(dataset)
print(f"Dataset loaded: {len(df)} samples")
print(df['label'].value_counts())
print(df.head(3))

In [ ]:
import pandas as pd
from tqdm import tqdm

# Take 100 fake + 100 real samples
fake_samples = df[df['label'] == 0].sample(100, random_state=42)
real_samples = df[df['label'] == 1].sample(100, random_state=42)
test_df = pd.concat([fake_samples, real_samples]).reset_index(drop=True)

print(f"Testing on {len(test_df)} samples (100 fake + 100 real)")
print("Running... this takes 5-10 minutes")

# Run batch testing
results = []
for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        headline = str(row['title'])[:512]  # limit length
        pred = news_detector(headline)[0]
        predicted = 0 if pred['label'] == 'LABEL_0' else 1
        results.append({
            'title': headline,
            'actual': row['label'],
            'predicted': predicted,
            'confidence': round(pred['score'] * 100, 2),
            'correct': row['label'] == predicted
        })
    except:
        pass

# Calculate metrics
results_df = pd.DataFrame(results)
accuracy = round(results_df['correct'].mean() * 100, 2)
fake_acc = round(results_df[results_df['actual']==0]['correct'].mean() * 100, 2)
real_acc = round(results_df[results_df['actual']==1]['correct'].mean() * 100, 2)

print(f"\n--- RESULTS ---")
print(f"Total tested : {len(results_df)}")
print(f"Overall accuracy : {accuracy}%")
print(f"Fake detection accuracy : {fake_acc}%")
print(f"Real detection accuracy : {real_acc}%")

# Save to CSV
results_df.to_csv('fake_news_results.csv', index=False)
print("\nSaved to fake_news_results.csv")

In [ ]:
from google.colab import files
from PIL import Image
from io import BytesIO
import pandas as pd

print("Select ALL 50 images at once (25 real + 25 fake)")
print("Hold Ctrl and click to select multiple files")
uploaded = files.upload()

print(f"\nUploaded {len(uploaded)} images")

In [ ]:
results = []

for filename, content in uploaded.items():
    try:
        image = Image.open(BytesIO(content)).convert("RGB")
        pred = img_detector(image)[0]
        label = pred['label'].lower()
        confidence = round(pred['score'] * 100, 2)

        # Fixed filename detection
        if 'whatsapp' in filename.lower():
            actual = 'real'
        elif 'download' in filename.lower():
            actual = 'fake'
        else:
            print(f"Skipping {filename} - can't determine actual label")
            continue

        correct = actual == label

        results.append({
            'filename': filename,
            'actual': actual,
            'predicted': label,
            'confidence': confidence,
            'correct': correct
        })
    except Exception as e:
        print(f"Error on {filename}: {e}")

results_df = pd.DataFrame(results)
accuracy = round(results_df['correct'].mean() * 100, 2)
fake_acc = round(results_df[results_df['actual']=='fake']['correct'].mean() * 100, 2)
real_acc = round(results_df[results_df['actual']=='real']['correct'].mean() * 100, 2)

print(f"--- RESULTS ---")
print(f"Total tested     : {len(results_df)}")
print(f"Overall accuracy : {accuracy}%")
print(f"Fake accuracy    : {fake_acc}%")
print(f"Real accuracy    : {real_acc}%")

results_df.to_csv('image_results.csv', index=False)
print("Saved to image_results.csv")

In [ ]:
# Check what filenames we have
for filename in uploaded.keys():
    print(filename)


In [ ]:
# Check 3 samples - 1 fake, 1 real
fake_file = 'download (1) (1).jfif'
real_file = 'WhatsApp Image 2026-05-27 at 2.53.32 PM.jpeg'

fake_img = Image.open(BytesIO(uploaded[fake_file])).convert("RGB")
real_img = Image.open(BytesIO(uploaded[real_file])).convert("RGB")

print("FAKE image result:")
print(img_detector(fake_img))

print("\nREAL image result:")
print(img_detector(real_img))

In [ ]:
results = []

for filename, content in uploaded.items():
    try:
        image = Image.open(BytesIO(content)).convert("RGB")
        pred = img_detector(image)[0]
        label = pred['label']  # 'Real' or 'Fake' - keep original case
        confidence = round(pred['score'] * 100, 2)

        if 'WhatsApp' in filename:
            actual = 'Real'
        elif 'download' in filename.lower():
            actual = 'Fake'
        else:
            print(f"Skipping: {filename}")
            continue

        correct = actual == label

        results.append({
            'filename': filename,
            'actual': actual,
            'predicted': label,
            'confidence': confidence,
            'correct': correct
        })
    except Exception as e:
        print(f"Error: {filename}: {e}")

results_df = pd.DataFrame(results)
accuracy = round(results_df['correct'].mean() * 100, 2)
fake_acc = round(results_df[results_df['actual']=='Fake']['correct'].mean() * 100, 2)
real_acc = round(results_df[results_df['actual']=='Real']['correct'].mean() * 100, 2)

print(f"--- RESULTS ---")
print(f"Total tested     : {len(results_df)}")
print(f"Overall accuracy : {accuracy}%")
print(f"Fake accuracy    : {fake_acc}%")
print(f"Real accuracy    : {real_acc}%")